In [1]:
# model-based-vacuumAgent

import tkinter as tk
import random
import time

# =========================
# CONFIG
# =========================
SIZE = 4
CELL_SIZE = 100
DELAY = 500            # milliseconds

DIRTY = 1
CLEAN = 0

# =========================
# MODEL-BASED VACUUM AGENT
# =========================
class VacuumAgent:

    def __init__(self, env):

        self.env = env

        self.x = 0
        self.y = 0

        self.visited_cells = set() # To track visited cells
        self.path_taken = [] # To track the path for potential backtracking/logic

        self.visited_cells.add((self.x, self.y))
        self.path_taken.append((self.x, self.y))

    # =========================
    # SENSOR
    # =========================
    def perceive(self):
        return self.env.grid[self.x][self.y]

    # =========================
    # CHECK POSSIBLE MOVES
    # =========================
    def get_possible_moves(self):
        moves = []
        if self.x > 0: moves.append(("UP", self.x - 1, self.y))
        if self.x < SIZE - 1: moves.append(("DOWN", self.x + 1, self.y))
        if self.y > 0: moves.append(("LEFT", self.x, self.y - 1))
        if self.y < SIZE - 1: moves.append(("RIGHT", self.x, self.y + 1))
        return moves

    # =========================
    # ACTUATOR + RULES
    # =========================
    def act(self):
        current_state = self.perceive()

        # RULE 1: If current cell is dirty, suck it
        if current_state == DIRTY:
            self.env.grid[self.x][self.y] = CLEAN
            print(f"SUCK at ({self.x}, {self.y})")
            return

        # RULE 2: If clean, decide next move based on visited cells
        # Get all physically possible moves
        all_possible_moves = self.get_possible_moves()

        # Filter out moves that lead to already visited cells
        unvisited_moves = [
            (action, next_x, next_y) for action, next_x, next_y in all_possible_moves
            if (next_x, next_y) not in self.visited_cells
        ]

        chosen_action = None
        if unvisited_moves:
            # Prioritize moving to an unvisited cell
            chosen_action = random.choice(unvisited_moves)
        else:
            # If all adjacent cells are visited, then pick a random possible move,
            # even if visited, to avoid getting completely stuck. This ensures
            # the agent can eventually reach all dirty cells.
            if all_possible_moves:
                chosen_action = random.choice(all_possible_moves)
            else:
                print("Agent is stuck! No possible moves.")
                return

        if chosen_action:
            action, next_x, next_y = chosen_action
            self.move(action, next_x, next_y)

    # =========================
    # MOVE
    # =========================
    def move(self, action, next_x, next_y):
        self.x = next_x
        self.y = next_y

        self.visited_cells.add((self.x, self.y))
        self.path_taken.append((self.x, self.y)) # Keep track of the path for debugging/advanced logic

        print(f"MOVE {action} -> ({self.x}, {self.y})")

    # =========================
    # CHECK ALL CLEAN
    # =========================
    def all_clean(self):
        for row in self.env.grid:
            for cell in row:
                if cell == DIRTY:
                    return False
        return True


# =========================
# ENVIRONMENT
# =========================
class Environment:

    def __init__(self):

        # Tạo ma trận random sạch/bẩn
        self.grid = [
            [random.choice([DIRTY, CLEAN]) for _ in range(SIZE)]
            for _ in range(SIZE)
        ]


# =========================
# GUI
# =========================
class VacuumGUI:

    def __init__(self, root):

        self.root = root
        self.root.title("Model-Based Vacuum Agent")

        self.canvas = tk.Canvas(
            root,
            width=SIZE * CELL_SIZE,
            height=SIZE * CELL_SIZE
        )

        self.canvas.pack()

        self.env = Environment()
        self.agent = VacuumAgent(self.env)

        self.draw_grid()

        # Start simulation
        self.root.after(DELAY, self.update)

    def draw_grid(self):

        self.canvas.delete("all")

        for i in range(SIZE):
            for j in range(SIZE):

                x1 = j * CELL_SIZE
                y1 = i * CELL_SIZE
                x2 = x1 + CELL_SIZE
                y2 = y1 + CELL_SIZE

                cell = self.env.grid[i][j]

                # Dirty = nâu
                if cell == DIRTY:
                    color = "saddle brown"
                else:
                    color = "white"

                self.canvas.create_rectangle(
                    x1, y1, x2, y2,
                    fill=color,
                    outline="black",
                    width=2
                )

                # Hiển thị trạng thái
                text = "DIRTY" if cell == DIRTY else "CLEAN"

                self.canvas.create_text(
                    x1 + CELL_SIZE // 2,
                    y1 + CELL_SIZE // 2,
                    text=text,
                    font=("Arial", 12, "bold")
                )

                # Mark visited cells (optional, for visualization)
                if (i, j) in self.agent.visited_cells:
                    self.canvas.create_oval(
                        x1 + CELL_SIZE // 4,
                        y1 + CELL_SIZE // 4,
                        x2 - CELL_SIZE // 4,
                        y2 - CELL_SIZE // 4,
                        fill="lightgray",
                        outline="gray"
                    )

        # Vẽ robot
        rx = self.agent.y * CELL_SIZE + CELL_SIZE // 2
        ry = self.agent.x * CELL_SIZE + CELL_SIZE // 2

        self.canvas.create_oval(
            rx - 25,
            ry - 25,
            rx + 25,
            ry + 25,
            fill="dodger blue"
        )

        self.canvas.create_text(
            rx,
            ry,
            text="Robot Hút Bụi",
            fill="white",
            font=("Arial", 16, "bold")
        )

    def update(self):

        if self.agent.all_clean():
            print("ALL CLEANED!")

            self.canvas.create_text(
                SIZE * CELL_SIZE // 2,
                SIZE * CELL_SIZE // 2,
                text="DONE",
                fill="green",
                font=("Arial", 30, "bold")
            )

            return

        self.agent.act()

        self.draw_grid()

        self.root.after(DELAY, self.update)


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # This part might cause TclError in environments without a display.
    # Consider handling this for Colab execution.
    try:
        root = tk.Tk()
        app = VacuumGUI(root)
        root.mainloop()
    except tk.TclError as e:
        print(f"Error: {e}. Tkinter applications often require a graphical display. "
              "This code might not run directly in some cloud environments without a display server."
              "The logic of the agent still works in the background.")
        # If GUI fails, we can still simulate the agent's actions in text.
        env = Environment()
        agent = VacuumAgent(env)
        print("\n--- Simulating agent without GUI ---")
        steps = 0
        max_steps = SIZE * SIZE * 5 # Arbitrary limit to prevent infinite loops if agent gets stuck
        while not agent.all_clean() and steps < max_steps:
            agent.act()
            steps += 1
            # Optional: print grid state after each action for text-based simulation
            # for row in agent.env.grid:
            #     print(row)
            # print(f"Agent at: ({agent.x}, {agent.y}), Visited: {agent.visited_cells}")
            # time.sleep(0.1) # small delay for readability
        if agent.all_clean():
            print(f"All cleaned in {steps} steps (text-based simulation).")
        else:
            print(f"Simulation ended after {steps} steps. Not all cells cleaned (text-based simulation).")


SUCK at (0, 0)
MOVE RIGHT -> (0, 1)
MOVE DOWN -> (1, 1)
SUCK at (1, 1)
MOVE RIGHT -> (1, 2)
MOVE UP -> (0, 2)
MOVE RIGHT -> (0, 3)
MOVE DOWN -> (1, 3)
SUCK at (1, 3)
MOVE DOWN -> (2, 3)
MOVE DOWN -> (3, 3)
SUCK at (3, 3)
MOVE LEFT -> (3, 2)
SUCK at (3, 2)
MOVE UP -> (2, 2)
MOVE LEFT -> (2, 1)
SUCK at (2, 1)
MOVE DOWN -> (3, 1)
SUCK at (3, 1)
ALL CLEANED!
